# 🛡️ Onyx-Nexus: Hepsi Bir Arada (All-in-One) Colab & Cloudflare Sunucusu
### 20GB RAM & GPU Gücüyle Bağımsız Çoklu Ajan ve OpenAI API Uç Noktası

- ⚡ **Tek Hücre:** Kurulum, Bağımlılıklar, FastAPI Sunucusu ve Cloudflare Tüneli tek bir tıkla ayağa kalkar.
- 🧠 **Sıfır Maliyet & Sıfır Key:** Awesome-FreeLLM-APIs (Pollinations DeepSeek-V3/R1 & OpenAI) ile tamamen ücretsiz LLM.
- 🛠️ **Akıllı Yetenekler:** Dinamik Araç Çağrısı (Tool Calling), Canlı WebSocket Terminal, Otomatik Bağımlılık Yükleyici, Akıllı Model Yönlendirici & Bellek Sıkıştırıcı.
- 🌐 **Open WebUI Uyumlu:** `https://....trycloudflare.com/v1` adresini Open WebUI veya herhangi bir OpenAI istemcisine bağlayabilirsiniz.

In [ ]:
# @title 🚀 Onyx-Nexus Otonom Sunucusunu Başlat (All-in-One / Tek Tıkla Çalıştır)
# @markdown Bu hücre; depoyu günceller, kütüphaneleri yükler, Cloudflare tünelini açar ve sunucuyu başlatır.

import os
import sys
import time
import json
import re
import shutil
import urllib.request
import subprocess
import threading

print("\033[1;35m=======================================================================\033[0m")
print("\033[1;35m   ONYX-NEXUS: GOOGLE COLAB HEPSİ BİR ARADA (ALL-IN-ONE) SUNUCU       \033[0m")
print("\033[1;35m=======================================================================\033[0m")

# 1. Adım: Depoyu Klonla veya Güncelle
REPO_URL = "https://github.com/furkanarslangray/onyx-nexus.git"
if not os.path.exists("main.py"):
    print("\033[1;36m[1/4] Onyx-Nexus deposu indiriliyor...\033[0m")
    subprocess.run(["git", "clone", REPO_URL, "."], check=False)
else:
    print("\033[1;36m[1/4] Depo güncelleniyor (git pull)...\033[0m")
    subprocess.run(["git", "pull"], check=False)

# 2. Adım: Gerekli Paketlerin Kurulumu
print("\033[1;36m[2/4] Python bağımlılıkları kuruluyor (FastAPI, Uvicorn, WebSockets, CrewAI)...\033[0m")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "fastapi>=0.110.0", "uvicorn[standard]>=0.28.0", "httpx>=0.27.0", "pydantic>=2.6.0",
    "websockets>=12.0", "crewai", "langchain-community", "langchain-openai", "psutil"
], check=False)

# 3. Adım: Cloudflared İndir ve Hazırla
print("\033[1;36m[3/4] Cloudflare Tunnel (cloudflared) hazırlanıyor...\033[0m")
cflared_bin = shutil.which("cloudflared") or "./cloudflared"
if not os.path.exists(cflared_bin) and not shutil.which("cloudflared"):
    cf_url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(cf_url, "./cloudflared")
    subprocess.run(["chmod", "+x", "./cloudflared"], check=True)
    cflared_bin = os.path.abspath("./cloudflared")

# 4. Adım: Uvicorn FastAPI Sunucusunu Başlat (Arka Plan)
print("\033[1;36m[4/4] FastAPI Motoru ve Cloudflare Tüneli başlatılıyor...\033[0m")
server_env = os.environ.copy()
server_env["EXECUTION_ENGINE"] = "colab"
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000", "--workers", "1"],
    env=server_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

def log_streamer():
    for line in iter(server_proc.stdout.readline, ''):
        if any(w in line for w in ["ERROR", "Uvicorn running", "ONYX-NEXUS", "AutonomousServer", "Inference"]):
            print(f"\033[0;34m[API]\033[0m {line.strip()}")
threading.Thread(target=log_streamer, daemon=True).start()
time.sleep(3)

# Cloudflare Tunnel Başlat ve Public URL'yi Yakala
tunnel_proc = subprocess.Popen(
    [cflared_bin, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
start_t = time.time()
while time.time() - start_t < 30:
    line = tunnel_proc.stdout.readline()
    if not line: break
    m = pattern.search(line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    # Sunucuya public URL bilgisini kaydet
    try:
        req = urllib.request.Request(
            "http://127.0.0.1:8000/api/set-tunnel",
            data=json.dumps({"url": public_url}).encode("utf-8"),
            headers={"Content-Type": "application/json"}
        )
        urllib.request.urlopen(req, timeout=2.0)
    except Exception:
        pass

    print("\n\033[1;32m" + "█" * 78)
    print("  🎉 ONYX-NEXUS ALL-IN-ONE SUNUCU BAŞARIYLA BAŞLATILDI!")
    print("█" * 78 + "\033[0m\n")
    print(f"\033[1;35m>>> 🖥️ COLAB WEB KONSOLU:\033[0m       \033[1;32m\033[4m{public_url}\033[0m")
    print(f"\033[1;36m>>> 🌐 OPEN WEBUI API BASE URL:\033[0m \033[1;32m{public_url}/v1\033[0m")
    print(f"\033[1;33m>>> 🔑 API KEY:\033[0m                 \033[1;33monyx-nexus-colab\033[0m")
    print(f"\033[1;36m>>> 🤖 MODEL ADI:\033[0m               \033[1;36monyx-nexus-agent\033[0m (veya onyx-nexus-deepseek)")
    print(f"\033[1;32m>>> ⚡ CANLI TERMİNAL WS:\033[0m        \033[1;32m{public_url.replace('https://', 'wss://')}/ws/terminal\033[0m")
    print("\033[1;35m-----------------------------------------------------------------------\033[0m")
    print("\033[1;37mSunucu arka planda aktif çalışıyor. Durdurmak için hücreyi durdurun.\033[0m\n")
else:
    print("\033[1;31m[!] Cloudflare tünel adresi yakalanamadı. Lütfen hücreyi tekrar çalıştırın.\033[0m")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\033[1;31m[-] Kapatılıyor...\033[0m")
    tunnel_proc.terminate()
    server_proc.terminate()
